# Can we predict failure within 90 days of an inspection?

Target: `failed_90d`. Plan: inspect the data, check each feature for leakage, then evaluate models with a **time-based split** (train on earlier inspections, test on the latest) against a base-rate baseline.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'missing:', int(df.isna().sum().sum()), 'dupes:', int(df.duplicated().sum()))
print('unique assets:', df.asset_tag.nunique(), '| date range:', df.inspected_at.min(), '->', df.inspected_at.max())
print('failure rate:', df.failed_90d.mean().round(4))
df.describe().T

## Leakage check
`parts_replaced` is the number of parts replaced on the work order *that closed this inspection record*, so it is only known after the inspection (and plausibly after the failure/repair). Let's see how it relates to the target.

In [ ]:
print(df.groupby('parts_replaced').failed_90d.agg(['mean','size']))

With 2+ parts replaced, **every** row failed (100%), and zero parts means no failure at all. That is near-deterministic: the column records the outcome rather than predicting it. It would not be available at inspection time, so **it is excluded from the predictive model**. Below we show how misleading including it would be.

Each `asset_tag` appears once, so there is no asset-level leakage between train and test.

## Features vs. target

In [ ]:
num = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig, ax = plt.subplots(1,5, figsize=(18,3))
for a,c in zip(ax,num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    df.groupby(q, observed=True).failed_90d.mean().plot(ax=a, marker='o'); a.set_title(c); a.set_xticks([]); a.set_ylim(0,None)
plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.mean().round(3))
mon = df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.agg(['mean','size'])
mon['mean'].plot(marker='o', title='Failure rate by inspection month'); plt.ylim(0,None); plt.show()
mon.T.round(3)

Check the decile plots for which features move the failure rate; site and ambient temperature should look flat if uninformative. The monthly plot shows whether late labels look censored (assets inspected in the final 90 days may not have had time to fail).

## Time-based split (train: first 75%, test: last 25%)

In [ ]:
feats = num + ['site']
cut = int(len(df)*0.75)
tr, te = df.iloc[:cut], df.iloc[cut:]
print('train ends', tr.inspected_at.max(), '| test starts', te.inspected_at.min(), '| test fail rate', te.failed_90d.mean().round(4))
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(), ['site'])])
models = {
 'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
 'grad_boost': make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
res = {}
fig, ax = plt.subplots(figsize=(5,5))
for n,m_ in models.items():
    m_.fit(tr[feats], tr.failed_90d); p = m_.predict_proba(te[feats])[:,1]; res[n]=p
    RocCurveDisplay.from_predictions(te.failed_90d, p, name=n, ax=ax)
ax.plot([0,1],[0,1],'k--'); plt.show()
pd.DataFrame({n:{'ROC AUC':roc_auc_score(te.failed_90d,p),'PR AUC':average_precision_score(te.failed_90d,p)} for n,p in res.items()}).T.assign(baseline_PR_AUC=te.failed_90d.mean()).round(3)

## Stability across time (5 expanding-window folds) and feature importance

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
rows=[]
for i,(a,b) in enumerate(TimeSeriesSplit(5).split(df)):
    for n,m_ in models.items():
        m_.fit(df.iloc[a][feats], df.iloc[a].failed_90d)
        p = m_.predict_proba(df.iloc[b][feats])[:,1]
        rows.append((i,n,roc_auc_score(df.iloc[b].failed_90d,p)))
cv = pd.DataFrame(rows, columns=['fold','model','auc']).pivot(index='fold',columns='model',values='auc'); print(cv.round(3)); print(cv.agg(['mean','std']).round(3))
pi = permutation_importance(models['grad_boost'], te[feats], te.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
pd.Series(pi.importances_mean, index=feats).sort_values().plot.barh(title='Permutation importance (test AUC drop)'); plt.show()

## The failure rate drifts over time, and `meter_hours` is a proxy for calendar time
The monthly failure rate swings from about 1% to 39% while the inspection features barely move. `meter_hours` grows steadily with calendar date, so its apparent (negative) link to failure may just reflect *when* the inspection happened. Compare each feature's AUC overall vs. within a single month:

In [ ]:
d = df.assign(month=df.inspected_at.dt.to_period('M'))
def within(c):
    return d.groupby('month')[[c,'failed_90d']].apply(lambda g: roc_auc_score(g.failed_90d, g[c]) if g.failed_90d.nunique()>1 else np.nan).mean()
pd.DataFrame({'AUC overall':{c:roc_auc_score(df.failed_90d, df[c]) for c in num},
              'AUC within month (mean)':{c:within(c) for c in num}}).round(3)

Vibration, bearing temperature and load keep their signal within a month; `meter_hours` falls to ~0.5, so it mostly encodes time. A model leaning on it can fit the past but will not carry into a period with a different base rate. Refit using only the physical measurements:

In [ ]:
core = ['vibration_rms','bearing_temp_c','load_pct']
pre3 = ColumnTransformer([('n', StandardScaler(), core)])
for n,clf in {'logistic':LogisticRegression(max_iter=1000),'grad_boost':HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)}.items():
    mm = make_pipeline(pre3, clf).fit(tr[core], tr.failed_90d); p = mm.predict_proba(te[core])[:,1]
    print(f'{n:11s} core features -> test ROC AUC {roc_auc_score(te.failed_90d,p):.3f}, PR AUC {average_precision_score(te.failed_90d,p):.3f} (base rate {te.failed_90d.mean():.3f})')

## Operational view: flag the top-k% riskiest inspections

In [ ]:
te2 = te.assign(p=res['grad_boost'])
rows=[]
for k in [5,10,20,30]:
    top = te2.nlargest(int(len(te2)*k/100),'p')
    rows.append((f'top {k}%', top.failed_90d.mean(), top.failed_90d.sum()/te2.failed_90d.sum()))
pd.DataFrame(rows, columns=['flagged','precision','recall']).round(3)

## What if we (wrongly) included `parts_replaced`?

In [ ]:
f2 = feats+['parts_replaced']
pre2 = ColumnTransformer([('n', StandardScaler(), num+['parts_replaced']), ('c', OneHotEncoder(), ['site'])])
leak = make_pipeline(pre2, HistGradientBoostingClassifier(max_depth=3, random_state=0)).fit(tr[f2], tr.failed_90d)
print('AUC with parts_replaced (leaky):', round(roc_auc_score(te.failed_90d, leak.predict_proba(te[f2])[:,1]),3))
print('AUC without (honest):          ', round(roc_auc_score(te.failed_90d, res['grad_boost']),3))

## Conclusions
**Answer: only weakly.** There is real but modest signal in the inspection measurements, not enough for a reliable per-asset predictor.

- **`parts_replaced` is leakage.** It records work done after the inspection and is ~deterministic with the target (>=2 parts -> 100% failed). Including it gives AUC ~0.997; it must not be used. (Worth confirming with the data owner when it is entered.)
- **Honest performance** (time split, last 25% as test, 4.3% failure rate): ROC AUC ~0.66-0.75, PR AUC 0.07-0.12 vs a 0.043 base rate. Flagging the top 10% of inspections catches ~31% of failures at ~13% precision (about 3x the base rate in that period).
- **Real signals:** vibration, bearing temperature and load (AUC ~0.63-0.68 even within a single month). Site and ambient temperature carry essentially nothing.
- **Caveat on the best model:** the gradient-boosting AUC (0.75) leans on `meter_hours`, which is just a proxy for calendar date (within-month AUC ~0.5). Using only vibration/temperature/load, test AUC is ~0.65-0.68. Treat ~0.68 as the trustworthy figure.
- **Non-stationary base rate:** failure rate swings from ~1% to ~39% by inspection month with no matching change in the features, so something outside the data (season, a batch, a maintenance campaign, label definition) drives much of the outcome. Absolute probabilities will not transfer across periods; the model is useful for ranking within a period, not for calibrated forecasts. Also check whether late-2025 labels are complete.
- **Next steps:** find the driver of the monthly swings; add history features (previous inspections per asset, trend in vibration), which need repeat inspections - every `asset_tag` here appears only once; and use calibration plus a cost-based threshold.
